# PornWorks Web Generator — Kaggle FIXED Notebook

Import this notebook into Kaggle. Turn Internet on and select a GPU accelerator (T4 is supported).

Run the setup cell, restart the session if Kaggle prompts you, then run the remaining cells in order. The model cell now verifies CUDA before loading and uses an SDXL-compatible scheduler to avoid the previous scheduler-config mismatch. The final cell prints a temporary Gradio URL that works only while the session remains active. Use only fictional consenting adults.


In [ ]:
# Remove the incompatible optional torchao package before importing diffusers.
!pip uninstall -y torchao
%pip install -q --upgrade diffusers accelerate safetensors gradio
# Ensure torchao was not reintroduced; it is optional for this pipeline.
!pip uninstall -y torchao

import importlib.util
if importlib.util.find_spec('torchao') is not None:
    raise RuntimeError('torchao is still present. Restart the Kaggle session, then rerun this setup cell.')
import torch, transformers, diffusers, gradio
from diffusers import DiffusionPipeline
print('Setup/imports OK')
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('transformers:', transformers.__version__)
print('diffusers:', diffusers.__version__)
print('gradio:', gradio.__version__)


In [ ]:
import torch
import gradio as gr
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler

MODEL_ID = "John6666/pornworks-real-porn-v03-sdxl"
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is selected but PyTorch cannot see CUDA. Stop this run, restart the Kaggle session, and rerun the notebook.")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))

# This SDXL model is stored in FP16; use FP16 on the T4 GPU.
pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)

# The repository declares an EDM DPM scheduler but its config contains standard
# beta-schedule fields that the EDM scheduler ignores. Use an SDXL-compatible
# scheduler that reads those fields instead of silently ignoring them.
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
print("Model loaded. Scheduler:", pipe.scheduler.__class__.__name__)
print("VAE dtype:", next(pipe.vae.parameters()).dtype)


In [ ]:
import random
import numpy as np
from pathlib import Path

def generate_image(prompt, negative_prompt, width, height, steps, guidance, seed):
    prompt = (prompt or "").strip()
    if not prompt:
        raise gr.Error("Enter a prompt first.")
    actual_seed = int(seed)
    if actual_seed < 0:
        actual_seed = random.randint(0, 2**31 - 1)
    generator = torch.Generator(device="cuda" if has_cuda else "cpu").manual_seed(actual_seed)
    try:
        with torch.inference_mode():
            result = pipe(prompt=prompt, negative_prompt=negative_prompt or None,
                          width=int(width), height=int(height),
                          num_inference_steps=int(steps), guidance_scale=float(guidance),
                          generator=generator, output_type="pil")
        image = result.images[0].convert("RGB")
        arr = np.asarray(image)
        if arr.size == 0 or not np.isfinite(arr).all() or arr.std() < 1.0:
            raise gr.Error("The model returned a blank/invalid image. Check logs and restart the session.")
        path = f"/kaggle/working/generated_{actual_seed}.png" if Path('/kaggle/working').exists() else f"/content/generated_{actual_seed}.png"
        image.save(path, format="PNG")
        return image, path, f"Seed: {actual_seed} | Size: {image.width}×{image.height}"
    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(f"Generation failed: {type(e).__name__}: {e}")

with gr.Blocks(theme=gr.themes.Soft(primary_hue="violet", neutral_hue="slate"), title="PornWorks Image Lab") as demo:
    gr.Markdown("# PornWorks Image Lab\nPrompt-to-image demo using the Real Porn V03 SDXL model. Use fictional consenting adults only.")
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label="Prompt", placeholder="Describe the scene, lighting, camera and style…", lines=5)
            negative = gr.Textbox(label="Negative prompt", value="pixelated, low quality, blurry, jpeg artifacts, distorted anatomy, extra fingers, watermark, text", lines=2)
            width = gr.Slider(512, 1024, value=768, step=64, label="Width")
            height = gr.Slider(512, 1024, value=768, step=64, label="Height")
            steps = gr.Slider(20, 50, value=32, step=1, label="Steps")
            guidance = gr.Slider(1, 10, value=5.0, step=0.5, label="Guidance scale")
            seed = gr.Number(value=-1, precision=0, label="Seed (-1 = random)")
            run = gr.Button("Generate image", variant="primary")
        with gr.Column():
            output = gr.Image(label="Generated image", type="pil", format="png")
            file_output = gr.File(label="Download PNG")
            info = gr.Textbox(label="Generation info", interactive=False)
    run.click(generate_image, inputs=[prompt, negative, width, height, steps, guidance, seed], outputs=[output, file_output, info])

demo.queue(max_size=2).launch(share=True, debug=True)

## Test prompt
`A photorealistic red apple on a wooden table, natural window light, realistic shadows, sharp focus, detailed texture, professional photography`

If generation fails, read the error displayed by Gradio and the notebook output. The model-loading cell must print CUDA available via the GPU name and `EulerAncestralDiscreteScheduler` as the active scheduler. If CUDA is not detected, restart the Kaggle session and rerun. Stop the session when finished to preserve GPU quota.